# Lab 02 guiado · Manipulación de datos

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 2, ruta guiada*

Este cuaderno se abre directo desde GitHub. Lo que escribas se pierde al cerrar la pestaña. Si quieres conservarlo, descárgalo con Archivo, Descargar, Descargar .ipynb.

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#31708f"></td><td bgcolor="#d9edf7"><font color="#0b3d62" size="3"><b>Qué vamos a responder</b><br>1. Cómo se eligen las filas que interesan.<br>2. Cómo se resume una tabla grande en pocas cifras.<br>3. Por qué una cifra global puede esconder lo que pasa en cada grupo.</font></td></tr></table>

---
<br>

## Paso 1 de 9 · ¿Con qué datos vamos a trabajar?

**Qué vamos a responder.** Qué tablas tenemos para este lab y de qué tamaño son.

**Qué hacer.** Corre la celda. Lee tres tablas y une la generación con la ficha de cada central, como en el Lab 01.

In [ ]:
# Paso 1
DATOS = "https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/datos/"
import pandas as pd

centrales = pd.read_csv(DATOS + "centrales.csv")
generacion = pd.read_csv(DATOS + "generacion.csv", parse_dates=["fecha"])
demanda = pd.read_csv(DATOS + "demanda.csv", parse_dates=["fecha"])

# La generación de cada hora con la ficha de su central al lado.
gen = pd.merge(generacion, centrales, on="central", how="left")
print("Generación con ficha", gen.shape)
print("Demanda             ", demanda.shape)

---
<br>

## Paso 2 de 9 · Tu turno, filtra con dos condiciones

**Qué vamos a responder.** Cuánto generó una tecnología en una región durante 2024.

**Qué hacer.** Elige una tecnología y una región en las listas y vuelve a correr la celda.

In [ ]:
# Paso 2
tecnologia = "solar" #@param ["hidro", "solar", "eolica", "gas", "carbon", "diesel"]
region = "Antofagasta" #@param ["Antofagasta", "Atacama", "Coquimbo", "Valparaíso", "Metropolitana", "Biobío", "Los Lagos"]

# El signo & se lee como "y". Cada condición va entre paréntesis.
elegidas = gen[(gen["tecnologia"] == tecnologia) & (gen["region"] == region)]

print("Filas encontradas", len(elegidas))
print(f"✓ Generación de {tecnologia} en {region}", round(elegidas["mwh"].sum()), "MWh")

---
<br>

## Paso 3 de 9 · ¿Se consume menos el fin de semana?

**Qué vamos a responder.** Cuánto baja la demanda el sábado y el domingo.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 3
# Una columna nueva que dice si cada fila cae en sábado o domingo.
# dayofweek numera los días de 0, lunes, a 6, domingo.
demanda["es_finde"] = demanda["fecha"].dt.dayofweek >= 5

promedio = demanda.groupby("es_finde")["mwh"].mean()
print(promedio.round(1))

caida = 100 * (1 - promedio[True] / promedio[False])
print(f"✓ El fin de semana se consume un {caida:.1f} por ciento menos")

---
<br>

## Paso 4 de 9 · ¿Cuánto generó cada tecnología en el año?

**Qué vamos a responder.** Qué tecnología generó más energía en 2024.

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 4
# Agrupar es lo mismo que hace una tabla dinámica.
# Junta las filas de cada tecnología y suma su generación.
por_tecnologia = gen.groupby("tecnologia")["mwh"].sum().round()
por_tecnologia.sort_values(ascending=False)

---
<br>

## Paso 5 de 9 · Tu turno, resume con varios cálculos

**Qué vamos a responder.** El total, el promedio por hora y cuántas centrales hay en cada grupo.

**Qué hacer.** Elige cómo agrupar en la lista y vuelve a correr la celda.

In [ ]:
# Paso 5
agrupar_por = "tecnologia" #@param ["tecnologia", "region"]

# Cada línea es una columna del resumen. A la izquierda el nombre,
# a la derecha la columna y el cálculo.
resumen = gen.groupby(agrupar_por).agg(
    total_mwh=("mwh", "sum"),
    promedio_hora=("mwh", "mean"),
    centrales=("central", "nunique"),
)
resumen.round(1).sort_values("total_mwh", ascending=False)

---
<br>

## Paso 6 de 9 · ¿Cuánto generó cada región con cada tecnología?

**Qué vamos a responder.** La generación del año cruzando región y tecnología.

**Qué hacer.** Corre la celda. Las regiones quedan en las filas y las tecnologías en las columnas.

In [ ]:
# Paso 6
# pivot_table arma una tabla de doble entrada, como una tabla dinámica.
# fill_value=0 pone cero donde una región no tiene esa tecnología.
cruce = gen.pivot_table(index="region", columns="tecnologia",
                        values="mwh", aggfunc="sum", fill_value=0)
cruce.round()

---
<br>

## Paso 7 de 9 · ¿Qué tan seguido genera cada tecnología?

**Qué vamos a responder.** El factor de planta promedio de cada tecnología.

**Qué hacer.** Corre la celda. El factor de planta es lo que generó una central dividido por lo máximo que podía generar.

In [ ]:
# Paso 7
# Una columna nueva calculada para las 175.680 filas de una vez.
gen["factor_planta"] = gen["mwh"] / gen["potencia_mw"]

factor = gen.groupby("tecnologia")["factor_planta"].mean().round(3)
factor.sort_values(ascending=False)

---
<br>

## Paso 8 de 9 · ¿Sube el precio cuando sube la demanda?

**Qué vamos a responder.** Si el precio y la demanda se mueven juntos, con todas las regiones a la vez.

**Qué hacer.** Corre la celda. Une los precios de enero con la demanda de enero.

In [ ]:
# Paso 8
import requests
respuesta = requests.get(DATOS + "precios_nudo.json").json()
precios = pd.json_normalize(respuesta["datos"])
precios["fecha"] = pd.to_datetime(precios["fecha"])

# Se unen por fecha, hora y región a la vez.
mix = pd.merge(precios, demanda, on=["fecha", "hora", "region"], how="inner")
print("Filas unidas", len(mix))

# La correlación va de -1 a 1. Cerca de 1 es que suben juntos.
print("✓ Correlación con todas las regiones", round(mix["precio_usd_mwh"].corr(mix["mwh"]), 3))

---
<br>

## Paso 9 de 9 · ¿Y dentro de cada región?

**Qué vamos a responder.** Si la relación entre precio y demanda se mantiene región por región.

**Qué hacer.** Elige una región en la lista y vuelve a correr la celda. Compara con el número del paso anterior.

In [ ]:
# Paso 9
region = "Metropolitana" #@param ["Antofagasta", "Atacama", "Coquimbo", "Valparaíso", "Metropolitana", "Biobío", "Los Lagos"]

una = mix[mix["region"] == region]
correlacion = round(una["precio_usd_mwh"].corr(una["mwh"]), 3)
print(f"Correlación dentro de {region}", correlacion)
print("Correlación con todas las regiones", round(mix["precio_usd_mwh"].corr(mix["mwh"]), 3))

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Un total puede engañar</b><br>Con todas las regiones juntas la correlación es 0,979, casi perfecta.<br>Dentro de cada región va de 0,125 a 0,917.<br>La relación fuerte aparece porque las regiones grandes tienen a la vez<br>más demanda y precios más altos.<br>Antes de informar una cifra global, se mira por grupo.</font></td></tr></table>

---
<br>

## Lo que lograste hoy

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#28a745"></td><td bgcolor="#d8f0df"><font color="#14532d" size="3">• Un filtro con & deja las filas que cumplen dos condiciones a la vez.<br>• Agrupar resume una tabla grande, igual que una tabla dinámica.<br>• Un resumen puede traer varios cálculos, y conviene que uno sea el conteo.<br>• Una columna nueva se calcula para todas las filas de una vez.<br>• Una cifra global puede esconder lo que pasa dentro de cada grupo.</font></td></tr></table>

Las fechas en detalle, tablas largas y anchas con melt y el trabajo con Polars están en [la ruta completa de este módulo](https://colab.research.google.com/github/RodrigoMSB/curso-python-minergia/blob/main/modulos/modulo02/lab.ipynb).